# Chapter 1.2 — From RNNs to ViT and CLIP

Chapter 1’s second half is the bridge from “vision as filters” to “vision as language-compatible tokens.”

You will:

1. Turn text into token ids.
2. Feel why an RNN is sequential and a transformer is not.
3. Patchify an image the way **ViT** does, then classify with Hugging Face.
4. Run **CLIP** zero-shot classification (image tower + text tower).
5. Peek at Hub abstractions: `pipeline`, `AutoModel`, `datasets`.

**Colab:** Runtime → GPU (T4) if you can. CPU works, just slower.

We use **CLIP ViT-B/32** so a free GPU stays comfortable. The book’s large CLIP checkpoint is the same idea with more parameters.


## 0. Setup


In [ ]:
import sys
import subprocess
import torch

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers",
        "datasets",
        "accelerate",
        "sentencepiece",
        "pillow",
        "matplotlib",
    ]
)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from io import BytesIO
from urllib.request import urlopen

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)

BEE_URL = "https://huggingface.co/datasets/vlmbook/images/resolve/main/bee.jpg"


def load_image(url: str) -> Image.Image:
    with urlopen(url, timeout=30) as f:
        return Image.open(BytesIO(f.read())).convert("RGB")


bee = load_image(BEE_URL)
plt.imshow(bee)
plt.axis("off")
plt.title("demo image")
plt.show()
print(bee.size)


## 1. Text becomes a sequence of integers

A tokenizer is a lookup table plus a splitting rule. The model never sees characters. It sees ids.

RNNs read those ids **one by one**. The hidden state is a running summary. That is great for short strings and painful for long ones: information has to survive many steps, and you cannot parallelize the time axis.

Transformers replace “pass the baton left-to-right” with **attention**: every token may look at every other token in one step.


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
sentence = "The bee landed on the flower."
encoded = tok(sentence)
print("tokens:", tok.convert_ids_to_tokens(encoded["input_ids"]))
print("ids:   ", encoded["input_ids"])

# Tiny RNN vs tiny self-attention on random embeddings (shapes only — intuition, not a trained LM).
torch.manual_seed(0)
ids = torch.tensor(encoded["input_ids"])
vocab, dim, hidden = tok.vocab_size, 32, 16
emb = torch.nn.Embedding(vocab, dim)
x = emb(ids).unsqueeze(1)  # [T, 1, D]

rnn = torch.nn.GRU(dim, hidden)
out_rnn, h_n = rnn(x)
print("RNN output steps", tuple(out_rnn.shape), "| final hidden", tuple(h_n.shape))

# One-head attention: Q, K, V from the same sequence (self-attention).
q = k = v = x.transpose(0, 1)  # [1, T, D]
scale = dim ** 0.5
attn = torch.softmax(q @ k.transpose(-1, -2) / scale, dim=-1)
mixed = attn @ v
print("self-attention weights", tuple(attn.shape), "(batch, query, key)")
print("each token mixed with all tokens:", tuple(mixed.shape))

plt.imshow(attn[0].detach().numpy(), cmap="magma")
plt.colorbar()
plt.xticks(range(len(encoded["input_ids"])), tok.convert_ids_to_tokens(encoded["input_ids"]), rotation=90)
plt.yticks(range(len(encoded["input_ids"])), tok.convert_ids_to_tokens(encoded["input_ids"]))
plt.title("Untrained self-attention (random weights)")
plt.tight_layout()
plt.show()


**What you should notice:** the RNN’s last hidden state is one vector for the whole sentence. Attention produces a **matrix** of “who looks at whom.” Random weights look noisy; trained BERT/GPT weights become structured.

Three pretraining games you will meet again:

| Style | Example | Game |
| --- | --- | --- |
| Masked (bidirectional) | BERT | hide a token, guess it from both sides |
| Autoregressive | GPT | guess the **next** token only from the left |
| Encoder–decoder | T5 | read an input sequence, generate an output sequence (translation, QA, instructions) |

---

## 2. Vision Transformer: the image is also a sequence

ViT does not slide a 3×3 forever. It:

1. Cuts the image into a grid of patches (classic: 224×224 image, 16×16 patches → 14×14 = **196** patches).
2. Flattens each patch into a vector and adds a position embedding.
3. Runs a transformer on that sequence, often with a `[CLS]` token whose output is the image embedding.

So an image is tokenized, just like a sentence.


In [ ]:
img = bee.resize((224, 224))
arr = np.array(img)
patch = 16
n = 224 // patch
fig, axes = plt.subplots(n, n, figsize=(6, 6))
for i in range(n):
    for j in range(n):
        axes[i, j].imshow(arr[i * patch : (i + 1) * patch, j * patch : (j + 1) * patch])
        axes[i, j].axis("off")
plt.suptitle(f"{n}x{n} = {n*n} patches of {patch}x{patch}")
plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.show()
print("sequence length if you add a CLS token:", n * n + 1)


### Hugging Face `pipeline`: model + processor in one box


In [ ]:
from transformers import pipeline

# Same checkpoint family as the book. Downloads on first run.
vit = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    device=0 if DEVICE == "cuda" else -1,
)
vit_preds = vit(bee)
for row in vit_preds:
    print(f"{row['score']:.3f}  {row['label']}")


**What you should notice:** ImageNet labels are closed-set (`bee`, `honeycomb`, `daisy`, …). The model cannot name a class it never saw. CLIP’s trick is to turn **any text string** into a class.

Related idea in the chapter (we will not download extra checkpoints here): **BEiT** masks visual tokens the way BERT masks words.

---

## 3. CLIP: one space for images and sentences

CLIP trains two encoders:

- image in → vector
- text in → vector

On a batch of paired captions, it **raises similarity for the true pairs** and lowers it for the mismatches (contrastive loss). After training, classification is retrieval: embed the photo, embed each candidate label, pick the nearest text.

**SigLIP** is a later CLIP-style model with a sigmoid loss instead of a softmax over the whole batch. Same user-facing trick: shared space + cosine similarity.


In [ ]:
from transformers import pipeline as hf_pipeline

# Book uses openai/clip-vit-large-patch14. Base/32 is the same API, lighter on Colab.
CLIP_ID = "openai/clip-vit-base-patch32"
clip = hf_pipeline(
    "zero-shot-image-classification",
    model=CLIP_ID,
    device=0 if DEVICE == "cuda" else -1,
)

labels = [
    "a bee on a flower",
    "a bee in a hive",
    "a dog in a park",
    "a car on a highway",
]
clip_preds = clip(bee, candidate_labels=labels)
for row in clip_preds:
    print(f"{row['score']:.3f}  {row['label']}")


### Same thing with `AutoProcessor` + `AutoModel` (what `pipeline` hides)


In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotImageClassification

processor = AutoProcessor.from_pretrained(CLIP_ID)
model = AutoModelForZeroShotImageClassification.from_pretrained(CLIP_ID).to(DEVICE)
model.eval()

inputs = processor(text=labels, images=bee, return_tensors="pt", padding=True)
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
with torch.no_grad():
    out = model(**inputs)
    probs = out.logits_per_image.softmax(dim=-1)[0].cpu()

print("logits_per_image shape", tuple(out.logits_per_image.shape), "← (images, texts)")
for label, p in zip(labels, probs):
    print(f"{p:.3f}  {label}")

# Tiny contrastive picture: similarity matrix between 2 phrases and this 1 image
# (extend by adding more images if you like).
fig, ax = plt.subplots(figsize=(6, 1.8))
ax.imshow(probs.numpy()[None, :], cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=25, ha="right")
ax.set_yticks([0])
ax.set_yticklabels(["bee.jpg"])
ax.set_title("softmax(image · text / temperature)")
plt.tight_layout()
plt.show()


**What you should notice:** the phrase that actually describes the photo wins. Swap in a dog photo and the ranking should flip. That ranking **is** zero-shot classification — no extra training on those label strings.

Hub reminder (look these up in the browser while the cell runs):

- **Model card** — what the model is for, metrics, intended use.
- **Files and versions** — weights, `config.json`, preprocessor.
- **Community** — issues and discussions.

`Trainer` is Hugging Face’s training loop wrapper (dataloaders, eval, checkpointing). Chapter 1 only introduces it; we do not fine-tune here.

---

## 4. `datasets`: one API for images, text, and both

`load_dataset` can stream from the Hub. Splits are `Dataset` objects. `map` is for preprocessing.


In [ ]:
from datasets import load_dataset

ds = load_dataset("merve/vqav2-small")
print(ds)
split = ds["validation"] if "validation" in ds else ds[list(ds.keys())[0]]
print("features:", split.features)
print("rows:", len(split))

row = split[0]
print("keys:", list(row.keys()))
for key in ("question", "multiple_choice_answer", "answers"):
    if key in row:
        print(f"{key}: {row[key]}")

# Show the image if the feature is a PIL image or a dict with bytes/path.
image = row.get("image")
if image is not None:
    if isinstance(image, dict) and "bytes" in image:
        image = Image.open(BytesIO(image["bytes"])).convert("RGB")
    plt.imshow(image)
    plt.axis("off")
    plt.title("VQA sample image")
    plt.show()


Optional: a `map` that adds question length. This is the same hook you will use later to tokenize or resize.


In [ ]:
def add_question_len(example):
    q = example.get("question") or ""
    example["question_len"] = len(q.split())
    return example


small = split.select(range(min(32, len(split))))
small = small.map(add_question_len)
print(small[0]["question"], "→", small[0]["question_len"], "words")
print("mean question_len", float(np.mean(small["question_len"])))


## Pause

You now have:

- a **closed-set** image classifier (ViT / ImageNet)
- an **open-set** classifier (CLIP + any label strings)
- Hub tools to load models and datasets without writing a training loop

Next notebook: put CLIP-style embeddings in **FAISS** and search paintings by text — the chapter’s first multimodal app.
